# 3. First XGBoost model

Can five jet summaries separate unpaired jets from simulated LLP jets? This is our first baseline. The later cell models are in the [analysis guide](../analysis/README.md).

Run with **PPP first look**. Saved results load by default; set `RETRAIN = True` to train again.

In [ ]:
from pathlib import Path
import json
import os
import sys
import pandas as pd
from IPython.display import display, Image

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "analysis/__init__.py").is_file())
sys.path.insert(0, str(PROJECT))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT / "analysis/.cache/matplotlib"))
from analysis.training.train_xgboost import run_experiment, score_saved_model

OUT = PROJECT / "analysis/outputs/xgboost_baseline"
RETRAIN = False
if RETRAIN or not (OUT / "metrics.json").is_file():
    run_experiment(PROJECT, OUT)

results = json.loads((OUT / "metrics.json").read_text())
audit = json.loads((OUT / "data_audit.json").read_text())
print("Files checked in this saved run:", len(audit["files"]))
print("All file hashes checked:", all(file["hash_verified"] for file in audit["files"]))

## Which jets?

- **Unpaired = 1:** our working BIB label; some contamination may remain.
- **LLP = 0:** only jets with `jet_is_llp_matched == 1`.
- Both: $p_T \geq 30$ GeV, $|\eta| < 1.7$, and $0 < f_{\mathrm{Tile}} < 1$.

The Tile cut keeps jets with energy in both calorimeters. It does not reproduce every LLP-matching condition. This baseline has no zero-vertex cut; that comes in the later studies.

In [ ]:
cutflow = pd.read_csv(OUT / "cutflow.csv")
stages = ["all_jets", "truth_selection", "finite_pt_eta_time", "pt_cut", "eta_cut", "mixed_calorimeters"]
display(cutflow.pivot(index="stage", columns="sample", values="jets").reindex(stages))

## Train, validate, test

Roughly **60%** trains the model, **20%** chooses the stopping point and score cut, and **20%** tests it. Jets from the same event stay together.

The cut aims to keep at least **95% of validation LLP jets**. Test retention can differ. IDs, filenames, truth flags and Emil's BIB score are not model inputs.

In [ ]:
display(pd.read_csv(OUT / "split_summary.csv"))
print("Events shared between splits:", audit["split"]["overlapping_event_groups"])

## Model inputs

| Input | Meaning |
|---|---|
| `jet_time_ATLAS` | Standard jet time (ns) |
| `jet_time_spread` | Stored time spread (ns) |
| `jet_time_sigma` | Estimated uncertainty on jet time (ns) |
| `jet_n_clusters` | Cluster count |
| `jet_TileEnergyFrac` | Fraction of energy in Tile |

XGBoost adds small decision trees that correct earlier errors. A higher score means more BIB-like, not a calibrated probability.

The other models below check timing alone, kinematics alone, and what changes without time uncertainty. The last check was added after the first results.

In [ ]:
def result_table(models):
    rows = []
    for name, model in models.items():
        test = model["test"]
        rows.append({"Model": name, "Test jets": test["n_jets"],
                     "ROC AUC": test["roc_auc"],
                     "Unpaired tagged (%)": 100 * test["bib_tag_fraction"],
                     "LLP kept (%)": 100 * test["llp_keep_fraction"]})
    return pd.DataFrame(rows).set_index("Model").round(3)

settings = results["training_parameters"]
print(f"Depth {settings['max_depth']}, learning rate {settings['learning_rate']}; "
      f"up to {settings['n_estimators']} trees, stopping after 40 rounds without improvement.")
display(result_table(results["models"]))

## Are the samples too different?

The kinematics-only model also works well. The samples have different $p_T$ distributions, and other inputs can reveal energy indirectly.

For a smaller check, we take equal class counts in narrow $p_T$ and $|\eta|$ bins. Training defines the allowed bins; each jet keeps its original split. This reduces one sample difference, not every difference between data and simulation.

In [ ]:
MATCHED = OUT / "matched_kinematics"
matched = results["matched_kinematics"]
matching_audit = json.loads((MATCHED / "matching_audit.json").read_text())
display(pd.DataFrame(matching_audit["event_summary"]))
display(Image(filename=str(MATCHED / "kinematics_before_after.png"), width=900))
display(result_table(matched["models"]))

## Reading the result

**Unpaired tagged** means removed as BIB. **LLP kept** means not removed. **ROC AUC** measures ranking: 0.5 is random, 1 is perfect.

The intervals below resample whole test events with a fixed score cut. They exclude uncertainty from training, the cut choice and the labels.

In [ ]:
test = matched["models"]["XGBoost"]["test"]
print(f"Test: {test['class_counts']['BIB']} unpaired and {test['class_counts']['LLP']} LLP jets.")
for key, label in [("bib_tag_fraction", "Unpaired tagged"), ("llp_keep_fraction", "LLP kept")]:
    interval = test[key + "_ci95"]
    print(f"{label}: {100 * test[key]:.1f}%")
    if interval is not None:
        low, high = interval
        print(f"  95% event-bootstrap interval: {100 * low:.1f}–{100 * high:.1f}%")
    else:
        print("  No informative interval for this sample.")
display(Image(filename=str(MATCHED / "test_discrimination.png"), width=1000))

## Delayed jets

Now check ATLAS time $>3$ ns with the **same score cut**. A high overall AUC does not guarantee good rejection here, and only a few delayed unpaired jets remain.

The next figure checks how performance changes with jet momentum and direction.

In [ ]:
delayed = matched["models"]["XGBoost"]["test_regions"]["delayed_time_gt_3ns"]
counts = delayed["confusion"]
print(f"Delayed unpaired tagged: {counts['BIB_tagged_BIB']} / {delayed['class_counts']['BIB']}")
print(f"Delayed LLP kept: {counts['LLP_kept_as_LLP']} / {delayed['class_counts']['LLP']}")
display(Image(filename=str(MATCHED / "test_kinematic_dependence.png"), width=1000))

## Two example jets

Take the first test jet from each class, chosen by row order, and score it with the saved five-input model.

In [ ]:
jets = pd.read_csv(MATCHED / "matched_jets.csv.gz")
examples = jets.loc[jets.split.eq("test")].groupby("label", sort=True).head(1).copy()
predictions = score_saved_model(examples, MATCHED)
columns = ["sample", "jet_pt_GeV", "jet_eta", "jet_time_ATLAS", "jet_time_spread",
           "jet_time_sigma", "jet_n_clusters", "jet_TileEnergyFrac"]
display(examples[columns].join(predictions).round(4))
print("BIB score cut:", test["threshold"])

## What this tells us

This baseline separates the samples, but part of that result comes from their different kinematics. The smaller check does not establish useful delayed-BIB rejection at this cut.

We have since trained cell models on zero-vertex unpaired data. See the [current comparison](../README.md#latest-test-on-run-484258). A new run is still needed for an independent test.

Code: [train_xgboost.py](../analysis/training/train_xgboost.py). Full local results: `analysis/outputs/xgboost_baseline/`.